# 05 - What staying in place saves: the data-gravity ledger and AI cost

**Business question:** what does it cost to move feature rows out to another platform and the scores back, and what did
the AI steps in this lab actually cost?

| Section | What it shows |
|---|---|
| 1 | One scoring task, two paths: round trip (rows out to pandas, scores back with `write_pandas`) vs in place (`MODEL(...)!PREDICT` in SQL), each under its own query tag |
| 2 | The ledger: rows out, rows in, boundary crossings, wall time |
| 3 | The same measurement on your own account: `ACCOUNT_USAGE` queries to size today's loop |
| 4 | AI cost: token estimates, where the cascade avoided calls, and actual usage from `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` |
| 5 | Spend controls, as reference SQL |

Runs standalone after `04`. Every section ends in a `CHECK`.

In [ ]:
NOTEBOOK_TAG = "transcript-churn-05"

# Session: Snowsight Workspaces supplies one; a local kernel (Cortex Code Desktop,
# ./demo.sh build) opens the connection saved in .env by ./demo.sh setup.
import json
import os
import time

import numpy as np
import pandas as pd

DB = "TRANSCRIPT_CHURN_ML_DEMO"
WH = "TRANSCRIPT_CHURN_ML_WH"
ROLE = "SYSADMIN"
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"
LLM = "claude-sonnet-4-5"
# The account this demo is allowed to build in. ./demo.sh setup writes it to .env;
# in Workspaces, replace the placeholder with your demo account name.
DEMO_ACCOUNT = os.environ.get("DEMO_EXPECTED_ACCOUNT", "<YOUR_DEMO_ACCOUNT>")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    LOCAL = False
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config(
        "connection_name", os.environ.get("SNOWFLAKE_DEFAULT_CONNECTION_NAME", "default")).create()
    LOCAL = True

session.use_role(ROLE)
session.use_warehouse(WH)
session.use_database(DB)
session.sql(f"ALTER SESSION SET QUERY_TAG = '{NOTEBOOK_TAG}'").collect()

if LOCAL:
    # Local kernels have no SQL cells: run each %%sql cell through this session instead.
    from IPython.core.magic import register_cell_magic
    from IPython.display import display

    @register_cell_magic
    def sql(line, cell):
        args = line.split()
        name = args[args.index("-r") + 1] if "-r" in args else None
        rows = session.sql(cell).collect()      # collect(): CALL and EXECUTE cannot be described first
        result = pd.DataFrame([r.as_dict() for r in rows])
        if name:
            get_ipython().user_ns[name] = result
        display(result.head(50))

CHECKS = {}


def check(section, ok, evidence):
    status = "PASS" if ok else "FAIL"
    CHECKS[section] = (status, evidence)
    print(f"CHECK {section}: {status} | {evidence}")
    if not ok:
        raise AssertionError(f"CHECK {section} failed: fix the cause and re-run this section")


def q(text):
    return session.sql(text).to_pandas()


def one(text):
    return session.sql(text).collect()[0][0]


account, role = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_ROLE()").collect()[0]
if account != DEMO_ACCOUNT:
    raise SystemExit(f"STOP: connected to {account}, but this demo may only build in {DEMO_ACCOUNT}. "
                     "Set DEMO_EXPECTED_ACCOUNT (./demo.sh setup does) or edit DEMO_ACCOUNT in this cell.")
if not session.sql(f"SHOW SCHEMAS LIKE 'RAW' IN DATABASE {DB}").collect():
    raise SystemExit(f"STOP: {DB}.RAW is missing. Run lab/setup.sql (./demo.sh setup) first.")
print(f"Connected as role {role} on {account}, warehouse {WH}, query tag {NOTEBOOK_TAG} "
      f"({'local kernel' if LOCAL else 'Workspaces session'})")

In [ ]:
if one(f"SELECT COUNT(*) FROM {DB}.ANALYTICS.CHURN_SCORES") == 0:
    raise SystemExit("STOP: ANALYTICS.CHURN_SCORES is empty. Run notebooks 01-04 first.")
SNAPSHOT = "2026-01-01"
FEATURES = [c["name"] for c in session.sql(f"DESCRIBE TABLE {DB}.FEATURES.SCORING_INPUT").collect()
            if c["name"] not in ("ACCOUNT_ID", "AS_OF")]
print(f"{len(FEATURES)} features per account; ledger scores the {SNAPSHOT} snapshot")

---
## 1. One scoring task, two paths

**Round trip** is the pattern of an external training and scoring platform: pull the feature rows to the client, score them
there, and write the scores back. **In place** runs the registered model where the rows already are. (The round trip trains its own model on the client, as an external platform would.) Each run sets its own
`QUERY_TAG`, so every statement it issued can be found afterwards.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

session.sql("ALTER SESSION SET QUERY_TAG = 'transcript-churn-ledger-roundtrip'").collect()
t0 = time.time()
train_pdf = session.table(f"{DB}.FEATURES.CHURN_TRAINING_SOURCE").to_pandas()                 # rows out (training)
score_pdf = session.sql(f"SELECT * FROM {DB}.FEATURES.SCORING_INPUT WHERE AS_OF = '{SNAPSHOT}'").to_pandas()  # rows out (scoring)
X = train_pdf[FEATURES].astype(float).fillna(0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X, train_pdf["EVENT"])
out = pd.DataFrame({"ACCOUNT_ID": score_pdf["ACCOUNT_ID"],
                    "CHURN_PROB": model.predict_proba(score_pdf[FEATURES].astype(float).fillna(0))[:, 1]})
session.write_pandas(out, "CHURN_SCORES_ROUNDTRIP", database=DB, schema="ANALYTICS", auto_create_table=True,
                     overwrite=True, table_type="")                                               # rows in
RT = {"rows_out": len(train_pdf) + len(score_pdf), "rows_in": len(out), "seconds": time.time() - t0,
      "values_out": (len(train_pdf) + len(score_pdf)) * len(FEATURES)}
print(f"round trip: {RT['rows_out']:,} rows ({RT['values_out']:,} feature values) out to the client, "
      f"{RT['rows_in']:,} scores back, {RT['seconds']:.1f} s")

In [ ]:
session.sql("ALTER SESSION SET QUERY_TAG = 'transcript-churn-ledger-inplace'").collect()
t0 = time.time()
args = ", ".join(FEATURES)
res = session.sql(f"""
CREATE OR REPLACE TABLE {DB}.ANALYTICS.CHURN_SCORES_INPLACE COMMENT = 'Ledger: scored where the rows live' AS
SELECT ACCOUNT_ID, {DB}.ML.CHURN_SURVIVAL!PREDICT({args}):CHURN_PROB::FLOAT AS CHURN_PROB
FROM {DB}.FEATURES.SCORING_INPUT WHERE AS_OF = '{SNAPSHOT}'""").collect()
IP = {"rows_out": 0, "rows_in": 0, "seconds": time.time() - t0, "values_out": 0,
      "scored": one(f"SELECT COUNT(*) FROM {DB}.ANALYTICS.CHURN_SCORES_INPLACE")}
session.sql(f"ALTER SESSION SET QUERY_TAG = '{NOTEBOOK_TAG}'").collect()
print(f"in place: {IP['scored']:,} accounts scored, 0 rows out, 0 rows back, {IP['seconds']:.1f} s")

In [ ]:
check("S1", RT["rows_out"] > 0 and IP["rows_out"] == 0 and IP["scored"] == RT["rows_in"],
      f"round trip moved {RT['rows_out']:,} rows out and {RT['rows_in']:,} back ({RT['seconds']:.1f} s); "
      f"in place scored {IP['scored']:,} with 0 rows moved ({IP['seconds']:.1f} s)")

---
## 2. The ledger

Query history for each tag shows what the client fetched: on the round trip, the result sets pulled to the client are the
rows out, and `write_pandas` then stages a file and loads the scores back. A **boundary crossing** is each time customer
rows leave the governed platform (to a client, a bucket, a container) and come back: every crossing is a copy to secure,
reconcile and pay egress for. The in-place run has none. Wall time is not the point at this size (2,000 rows; in our
build the round trip was faster); rows moved and copies created are.

In [ ]:
hist = q(f"""
SELECT QUERY_TAG, QUERY_TYPE, COUNT(*) AS STATEMENTS, SUM(ROWS_PRODUCED) AS ROWS_PRODUCED,
       ROUND(SUM(BYTES_WRITTEN_TO_RESULT) / 1e6, 2) AS MB_TO_CLIENT
FROM TABLE({DB}.INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 1000))
WHERE QUERY_TAG IN ('transcript-churn-ledger-roundtrip', 'transcript-churn-ledger-inplace')
GROUP BY 1, 2 ORDER BY 1, 2""")
print(hist.to_string(index=False))
LEDGER = pd.DataFrame([
    {"PATH": "Round trip (out to pandas, back with write_pandas)", "ROWS_OUT": RT["rows_out"],
     "FEATURE_VALUES_OUT": RT["values_out"], "ROWS_BACK": RT["rows_in"], "BOUNDARY_CROSSINGS": 2,
     "SECONDS": round(RT["seconds"], 1)},
    {"PATH": "In place (MODEL(...)!PREDICT in SQL)", "ROWS_OUT": 0, "FEATURE_VALUES_OUT": 0, "ROWS_BACK": 0,
     "BOUNDARY_CROSSINGS": 0, "SECONDS": round(IP["seconds"], 1)}])
session.write_pandas(LEDGER, "DATA_GRAVITY_LEDGER", database=DB, schema="ANALYTICS", auto_create_table=True,
                     overwrite=True, table_type="")
LEDGER

In [ ]:
rt_tag = hist[hist.QUERY_TAG == "transcript-churn-ledger-roundtrip"]
ip_tag = hist[hist.QUERY_TAG == "transcript-churn-ledger-inplace"]
mb_out = float(rt_tag.MB_TO_CLIENT.fillna(0).sum())
check("S2", len(rt_tag) > 0 and len(ip_tag) > 0 and mb_out > 0,
      f"round trip: {int(rt_tag.STATEMENTS.sum())} statements, {mb_out:.2f} MB of results to the client, query types "
      f"{sorted(rt_tag.QUERY_TYPE)}; in place: {int(ip_tag.STATEMENTS.sum())} statements, "
      f"{float(ip_tag.MB_TO_CLIENT.fillna(0).sum()):.2f} MB to the client")

---
## 3. Size today's loop on your own account

The same measurement works on the account that runs today's loop. Replace the user (or set a query tag on the scheduler's
connection) and the warehouse; `ACCOUNT_USAGE` views lag by up to about 45 minutes. The query below runs here on this demo's
own tags.

In [ ]:
%%sql -r account_usage_ledger
-- Rows fetched by a pipeline identity and rows it loaded back, per day, last 30 days.
-- Replace the QUERY_TAG filter with USER_NAME = '<scheduler service user>' for an existing loop.
SELECT DATE_TRUNC(day, START_TIME)::DATE AS DAY,
       SUM(IFF(QUERY_TYPE = 'SELECT', ROWS_PRODUCED, 0)) AS ROWS_FETCHED,
       SUM(IFF(QUERY_TYPE IN ('INSERT', 'COPY', 'MERGE'), ROWS_INSERTED, 0)) AS ROWS_LOADED_BACK,
       COUNT(*) AS STATEMENTS
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND QUERY_TAG LIKE 'transcript-churn-ledger-%'
GROUP BY 1 ORDER BY 1

In [ ]:
check("S3", isinstance(account_usage_ledger, pd.DataFrame),
      f"ACCOUNT_USAGE.QUERY_HISTORY query ran ({len(account_usage_ledger)} day rows so far; the view lags up to ~45 min)")

---
## 4. What the AI steps cost

`AI_COUNT_TOKENS` estimates input tokens before a full pass. `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY`
records the credits each AI function call actually consumed, with the query tag, so this lab's spend can be read back per
notebook (it also lags). Credit rates per million tokens are in the Snowflake Service Consumption Table.

In [ ]:
est = q(f"""
SELECT 'AI_EMBED (all segments)' AS STEP, COUNT(*) AS CALLS,
       SUM(AI_COUNT_TOKENS('ai_embed', 'snowflake-arctic-embed-l-v2.0', REDACTED_TEXT)) AS INPUT_TOKENS
FROM {DB}.CORE.SEGMENT_EMBEDDINGS
UNION ALL
SELECT 'AI_SENTIMENT (prefiltered)', COUNT(*), SUM(AI_COUNT_TOKENS('ai_sentiment', CUSTOMER_TEXT))
FROM {DB}.ANALYTICS.PRICE_SENTIMENT
UNION ALL
SELECT 'AI_COMPLETE (cluster names)', COUNT(*), NULL FROM {DB}.ML.PROPOSED_TOPICS""")
segments = one(f"SELECT COUNT(*) FROM {DB}.CORE.SEGMENTS")
dials = one(f"SELECT COUNT_IF(CALL_TYPE = 'OUTBOUND_DIAL') FROM {DB}.RAW.CALLS")
AVOIDED = {"short_dials_never_segmented": dials,
           "segments_skipped_by_price_prefilter": segments - int(est.loc[1, "CALLS"]),
           "llm_calls_per_cluster_not_per_segment": f"{int(est.loc[2, 'CALLS'])} instead of "
                                                    f"{one(f'SELECT COUNT(*) FROM {DB}.ML.CLUSTER_MEMBERS')}"}
print(est.to_string(index=False))
print(AVOIDED)

In [ ]:
%%sql -r ai_usage_history
SELECT QUERY_TAG, FUNCTION_NAME, MODEL_NAME, COUNT(*) AS QUERIES, ROUND(SUM(CREDITS), 4) AS CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD(day, -7, CURRENT_TIMESTAMP()) AND QUERY_TAG LIKE 'transcript-churn-%'
GROUP BY 1, 2, 3 ORDER BY CREDITS DESC

In [ ]:
EMBED_TOKENS = int(est.loc[0, "INPUT_TOKENS"])
check("S4", EMBED_TOKENS > 0 and int(est.loc[1, "CALLS"]) < segments,
      f"estimated input tokens: embed {EMBED_TOKENS:,}, sentiment {int(est.loc[1, 'INPUT_TOKENS']):,}; avoided {AVOIDED}; "
      f"usage history rows so far {len(ai_usage_history)} (lags)")

---
## 5. Spend controls (reference, not run)

The controls that stop a full re-embed or an unbounded LLM pass from surprising anyone. Run them as a role with the
required privileges; names in angle brackets are yours.

```sql
-- A per-user daily cap on AI function credits, enforced (queries are blocked once the cap is reached)
CREATE SNOWFLAKE.CORE.QUOTA <db>.<schema>.AI_DAILY_CAP();
CALL <db>.<schema>.AI_DAILY_CAP!ADD_SHARED_RESOURCE('AI FUNCTION');
CALL <db>.<schema>.AI_DAILY_CAP!SET_PER_USER_LIMIT(<credits>, 'DAILY');
CALL <db>.<schema>.AI_DAILY_CAP!SET_BLOCK_ENFORCEMENT_ENABLED(TRUE);

-- A custom budget over the pipeline's warehouse and objects, with a notification
CREATE SNOWFLAKE.CORE.BUDGET <db>.<schema>.TRANSCRIPT_PIPELINE_BUDGET();
CALL <db>.<schema>.TRANSCRIPT_PIPELINE_BUDGET!SET_SPENDING_LIMIT(<credits per month>);
CALL <db>.<schema>.TRANSCRIPT_PIPELINE_BUDGET!ADD_RESOURCE(
  SYSTEM$REFERENCE('WAREHOUSE', 'TRANSCRIPT_CHURN_ML_WH', 'SESSION', 'APPLYBUDGET'));
CALL <db>.<schema>.TRANSCRIPT_PIPELINE_BUDGET!SET_EMAIL_NOTIFICATIONS('<integration>', '<you@company.com>');
```

Plus the habits used in this lab: estimate with `AI_COUNT_TOKENS` before a full pass, keep AI functions in incremental
pipelines so each row is processed once, filter before calling a model, and call an LLM per cluster rather than per row.

In [ ]:
check("S5", True, "spend-control reference SQL documented (quota, budget); not executed in the lab")
print("\n".join(f"{k}: {v[0]}" for k, v in CHECKS.items()))

---
## Summary

Scoring in place moved no rows; the round trip moved every training and scoring row out and the scores back. The AI steps
cost what the token estimates said, because short dials, non-price segments and per-segment LLM calls were never sent.